In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option("display.expand_frame_repr", False)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
from sentence_transformers import SentenceTransformer
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model = SentenceTransformer("all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

DATA_DIR = "/content/drive/MyDrive/CS 489" 
FILE = os.path.join(DATA_DIR, "CS 489 - Self Playing - Data Collection (1).csv")

Mounted at /content/drive/


In [ ]:

# Separate dataset into 75/10/15 split.
TRAIN_SPLIT = 0.75 
VAL_SPLIT   = 0.10
TEST_SPLIT  = 0.15
df          = pd.read_csv(FILE)
train_parts, val_parts, test_parts = [], [], []

# Scan through all unique word pairs (30).
for pair_id, group in df.groupby("anchor_left (0)"):
    
    # Extract 15% of total dataset into test set. 
    train_and_val, test = train_test_split(
        group, test_size = TEST_SPLIT, random_state = RANDOM_SEED)
    
    # Extract 10% of remaining 85% for validation split.
    train, val = train_test_split(
        train_and_val, test_size = VAL_SPLIT / (1 - TEST_SPLIT), random_state = RANDOM_SEED)

    train_parts.append(train)
    val_parts.append(val)
    test_parts.append(test)

# Turn train/val/test sets back into singular dfs.
train_df = pd.concat(train_parts).reset_index(drop = True)
val_df   = pd.concat(val_parts).reset_index(drop = True)
test_df  = pd.concat(test_parts).reset_index(drop = True)

print(train_df)
print(val_df)
print(test_df)


         anchor_left (0)  anchor_right (100)                     clue  target response_list response_mean  nathan kim  dp    category difference
0                    Bad                Good          Cutting in line    34.0            33            33         NaN NaN  Subjective          1
1                    Bad                Good           Curing disease    91.0            92            92         NaN NaN  Subjective          1
2                    Bad                Good       Fostering children    70.0            90            90         NaN NaN  Subjective         20
3                    Bad                Good        Community service    76.0            75            75         NaN NaN  Subjective          1
4                    Bad                Good         Selfless heroism   100.0            90            90         NaN NaN  Subjective         10
5                    Bad                Good     Forgetting birthdays    34.0            23            23         NaN NaN  Subject

In [ ]:

def cosine_similarity(a, b):
    dot    = np.sum(a * b, axis=1)
    norm_a = np.linalg.norm(a, axis=1)
    norm_b = np.linalg.norm(b, axis=1)
    return dot / (norm_a * norm_b)

def semantic_axis_score(left_enc, right_enc, clue_enc):
    axis          = right_enc - left_enc
    clue_relative = clue_enc  - left_enc
    axis_norm_sq  = np.sum(axis ** 2, axis=1)
    t             = np.sum(clue_relative * axis, axis=1) / axis_norm_sq
    return t

# Extract strings from train df
left_words  = np.array(train_df.loc[:, "anchor_left (0)"])
right_words = np.array(train_df.loc[:, "anchor_right (100)"])
clue_words  = np.array(train_df.loc[:, "clue"])

# Clean NaN values
left_words  = np.array([str(w) if isinstance(w, str) else "" for w in left_words])
right_words = np.array([str(w) if isinstance(w, str) else "" for w in right_words])
clue_words  = np.array([str(w) if isinstance(w, str) else "" for w in clue_words])

# Richer prompt templates
left_prompts  = "Something that is clearly " + left_words  + " in nature"
right_prompts = "Something that is clearly " + right_words + " in nature"
clue_prompts  = "The concept or activity known as: "        + clue_words

# Encode strings numerically via sentence transformer
left_enc  = model.encode(left_prompts,  normalize_embeddings=True)
right_enc = model.encode(right_prompts, normalize_embeddings=True)
clue_enc  = model.encode(clue_prompts,  normalize_embeddings=True)

# Cosine similarities
left_cos_sim  = cosine_similarity(left_enc,  clue_enc)
right_cos_sim = cosine_similarity(right_enc, clue_enc)

# Improved scores
delta  = right_cos_sim - left_cos_sim          # positive = closer to right anchor
scores = semantic_axis_score(left_enc, right_enc, clue_enc)  # 0.0 = left, 1.0 = right

for lword, rword, clue, lcs, rcs, d, s in zip(
    left_words, right_words, clue_words,
    left_cos_sim, right_cos_sim, delta, scores
):
    print(f"\n{lword} / {rword} -- {clue}")
    print(f"  left_cos: {lcs:.4f} | right_cos: {rcs:.4f} | delta: {d:.4f} | axis_score: {s:.4f}")


Bad / Good -- Cutting in line
  left_cos: 0.1694 | right_cos: 0.0737 | delta: -0.0957 | axis_score: 0.3476

Bad / Good -- Curing disease
  left_cos: 0.2235 | right_cos: 0.2697 | delta: 0.0462 | axis_score: 0.5736

Bad / Good -- Fostering children
  left_cos: 0.1912 | right_cos: 0.1593 | delta: -0.0319 | axis_score: 0.4491

Bad / Good -- Community service
  left_cos: 0.1506 | right_cos: 0.1644 | delta: 0.0138 | axis_score: 0.5220

Bad / Good -- Selfless heroism
  left_cos: 0.1132 | right_cos: 0.1201 | delta: 0.0068 | axis_score: 0.5108

Bad / Good -- Forgetting birthdays
  left_cos: 0.1337 | right_cos: -0.0053 | delta: -0.1390 | axis_score: 0.2787

Bad / Good -- Feeding homeless
  left_cos: 0.2075 | right_cos: 0.1781 | delta: -0.0294 | axis_score: 0.4532

Bad / Good -- Adopting a pet
  left_cos: 0.2185 | right_cos: 0.1607 | delta: -0.0578 | axis_score: 0.4079

Bad / Good -- Helping a stranger
  left_cos: 0.1633 | right_cos: 0.1510 | delta: -0.0123 | axis_score: 0.4804

Bad / Good -- Th

In [ ]:
def axis_projection_predict(left_enc, right_enc, clue_enc):
    axis       = right_enc - left_enc          # direction from left → right
    clue_shift = clue_enc  - left_enc          # clue relative to left pole
    
    # Scalar projection of clue onto the axis
    proj = np.sum(clue_shift * axis, axis=1) / (np.linalg.norm(axis, axis=1) ** 2 + 1e-8)
    return np.clip(proj, 0, 1) * 100

# Convert the two cos_sim scores into a single [0, 100] prediction
def predict_position(left_sim, right_sim):
    total = left_sim + right_sim
    return (right_sim / total) * 100

# Utilize combined MAE as a loss.
predictions = predict_position(left_cos_sim, right_cos_sim)
mae = np.mean(np.abs(predictions - pd.to_numeric(train_df["response_mean"], errors="coerce")))
print(f"Baseline MAE: {mae:.2f}")

predictions = axis_projection_predict(left_enc, right_enc, clue_enc)
mae = np.mean(np.abs(predictions - pd.to_numeric(train_df["response_mean"], errors="coerce")))
print(f"Baseline MAE adjusted: {mae:.2f}")

Baseline MAE: 25.19
Baseline MAE adjusted: 23.16


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from scipy.stats import pearsonr

# ── 1. Dataset ────────────────────────────────────────────────
class WavelengthDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=64, augment=False):
        self.df = df.reset_index(drop=True)
        self.df["response_mean"] = pd.to_numeric(self.df["response_mean"], errors="coerce")
        self.df = self.df.dropna(subset=["response_mean"])
        self.tokenizer = tokenizer
        self.max_len   = max_length
        self.augment   = augment  # flip left/right anchors → inverted label

    def __len__(self):
        return len(self.df) * (2 if self.augment else 1)

    def __getitem__(self, idx):
        flip   = self.augment and idx >= len(self.df)
        row    = self.df.iloc[idx % len(self.df)]
        left   = row["anchor_left (0)"]
        right  = row["anchor_right (100)"]
        label  = row["response_mean"] / 100.0

        if flip:
            left, right = right, left   # swap anchors
            label = 1.0 - label         # invert label

        text = f"clue: {row['clue']} | left: {left} | right: {right}"
        enc  = self.tokenizer(
            text,
            max_length     = self.max_len,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(label, dtype=torch.float32)
        }


# ── 2. Model ──────────────────────────────────────────────────
class WallyRegressor(nn.Module):
    def __init__(self, model_name, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        for param in self.encoder.parameters():
            param.requires_grad = False

        # Unfreeze last 4 layers instead of 2
        for layer in self.encoder.encoder.layer[-4:]:
            for param in layer.parameters():
                param.requires_grad = True

        hidden = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def mean_pool(self, token_embeddings, attention_mask):
        # Mean pooling — correct for mpnet, much better than CLS
        mask_expanded = attention_mask.unsqueeze(-1).float()
        return (token_embeddings * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pool(out.last_hidden_state, attention_mask)
        return self.head(pooled).squeeze(-1)


# ── 3. Setup ──────────────────────────────────────────────────
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
EPOCHS     = 40
BATCH_SIZE = 16
LR         = 2e-5
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = WavelengthDataset(train_df, tokenizer, augment=True)  # 2x data via flip
val_ds   = WavelengthDataset(val_df,   tokenizer, augment=False)
test_ds  = WavelengthDataset(test_df,  tokenizer, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

wally     = WallyRegressor(MODEL_NAME).to(DEVICE)
criterion = nn.L1Loss()   # MAE loss — directly optimizes what we measure
optimizer = AdamW(
    filter(lambda p: p.requires_grad, wally.parameters()),
    lr=LR, weight_decay=1e-2
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = len(train_loader) * 2,
    num_training_steps = len(train_loader) * EPOCHS
)


# ── 4. Train + Validate ───────────────────────────────────────
best_val_mae = float("inf")

for epoch in range(EPOCHS):

    # — Train —
    wally.train()
    train_loss = 0
    for batch in train_loader:
        ids    = batch["input_ids"].to(DEVICE)
        mask   = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        preds = wally(ids, mask)
        loss  = criterion(preds, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(wally.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    # — Validate —
    wally.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids    = batch["input_ids"].to(DEVICE)
            mask   = batch["attention_mask"].to(DEVICE)
            preds  = wally(ids, mask).cpu().numpy() * 100
            labels = batch["label"].numpy()          * 100
            val_preds.extend(preds)
            val_labels.extend(labels)

    val_mae = np.mean(np.abs(np.array(val_preds) - np.array(val_labels)))
    print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss/len(train_loader):.4f} | Val MAE: {val_mae:.2f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        torch.save(wally.state_dict(), "wally_best.pt")
        print(f"           ↑ New best saved")


# ── 5. Test Evaluation ────────────────────────────────────────
wally.load_state_dict(torch.load("wally_best.pt", map_location=DEVICE))
wally.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        ids    = batch["input_ids"].to(DEVICE)
        mask   = batch["attention_mask"].to(DEVICE)
        preds  = wally(ids, mask).cpu().numpy() * 100
        labels = batch["label"].numpy()          * 100
        test_preds.extend(preds)
        test_labels.extend(labels)

test_preds  = np.array(test_preds)
test_labels = np.array(test_labels)
test_mae    = np.mean(np.abs(test_preds - test_labels))
r, _        = pearsonr(test_preds, test_labels)

print(f"\n── Final Test Results ──────────────────")
print(f"Test MAE:              {test_mae:.2f}")
print(f"Pearson r:             {r:.3f}")
print(f"\n── Zero-shot Baselines ─────────────────")
print(f"Cosine ratio MAE:      25.19")
print(f"Axis projection MAE:   23.16")

Training on: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch   1 | Train Loss: 0.2815 | Val MAE: 28.63
           ↑ New best saved
Epoch   2 | Train Loss: 0.2810 | Val MAE: 28.75
Epoch   3 | Train Loss: 0.2813 | Val MAE: 28.33
           ↑ New best saved
Epoch   4 | Train Loss: 0.2814 | Val MAE: 28.32
           ↑ New best saved
Epoch   5 | Train Loss: 0.2809 | Val MAE: 28.29
           ↑ New best saved
Epoch   6 | Train Loss: 0.2804 | Val MAE: 28.31
Epoch   7 | Train Loss: 0.2807 | Val MAE: 28.50
Epoch   8 | Train Loss: 0.2809 | Val MAE: 28.48
Epoch   9 | Train Loss: 0.2806 | Val MAE: 28.30
Epoch  10 | Train Loss: 0.2777 | Val MAE: 27.18
           ↑ New best saved
Epoch  11 | Train Loss: 0.2726 | Val MAE: 27.06
           ↑ New best saved
Epoch  12 | Train Loss: 0.2670 | Val MAE: 25.85
           ↑ New best saved
Epoch  13 | Train Loss: 0.2616 | Val MAE: 25.05
           ↑ New best saved
Epoch  14 | Train Loss: 0.2554 | Val MAE: 24.59
           ↑ New best saved
Epoch  15 | Train Loss: 0.2485 | Val MAE: 24.32
           ↑ New best saved
